# 15 · Release a document parser

**Question.** Which training and serving changes meet a predeclared release gate?

Rebuild the supervised invoice parser on the same catalogue and base revision. Compare the rules baseline, prompted vision model, supervised adapter and verifiable-reward adapter. Then benchmark generation separately and restore the incumbent.

[Open the lesson](https://docs.recohut.com/courses/advanced-llm-fine-tuning/4-release/15-release-a-document-parser).

## Which software and hardware does this lab use?

The same quantised base and processor serve every branch. Only language-backbone adapters train. The notebook stops early if CUDA is unavailable.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.51.3", "peft==0.15.2", "accelerate==1.6.0",
                "bitsandbytes==0.50.2", "pillow==11.3.0", "datasets==3.6.0",
                "pypdfium2==4.30.0"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'peft==0.15.2', 'accelerate==1.6.0', 'bitsandbytes==0.50.2', 'pillow==11.3.0', 'datasets==3.6.0', 'pypdfium2==4.30.0'], returncode=0)

## How are invoice families held out?

Original synthetic records are split by supplier-and-layout family before rendering. The same example retains its family identifier across image derivatives.

In [2]:
"""Versioned synthetic invoices and independent record scoring for the course."""

from __future__ import annotations

import json
import random
import re
from collections import Counter
from dataclasses import dataclass
from datetime import date, timedelta
from typing import Any

CATALOG_VERSION = "course-invoice-v1"
SPLIT_SIZES = {"train": 576, "dev": 72, "familiar": 96, "novel": 96, "independent": 48, "release": 120}
VISIBLE_CONTRACT = (
    "Extract exactly one JSON object with invoice_id, issued_on (YYYY-MM-DD), supplier, "
    "currency (ISO code), line_items (description, quantity, unit_minor), subtotal_minor, "
    "tax_minor, and total_minor. Preserve line order and convert currency amounts to integer cents. "
    "Use null only for a missing or ambiguous field. Do not rewrite printed amounts, invent fields, or add explanations."
)

@dataclass(frozen=True)
class Line:
    description: str
    quantity: int
    unit_minor: int


@dataclass(frozen=True)
class Invoice:
    key: str
    family: str
    supplier: str
    issued_on: str
    currency: str
    items: tuple[Line, ...]
    tax_minor: int | None
    locale: str
    split: str
    layout: str = "register"
    printed_total_minor: int | None = None


def invoice_record(invoice: Invoice) -> dict[str, Any]:
    subtotal = sum(line.quantity * line.unit_minor for line in invoice.items)
    total = subtotal + invoice.tax_minor if invoice.tax_minor is not None else None
    if invoice.printed_total_minor is not None:
        total = invoice.printed_total_minor
    return {
        "invoice_id": invoice.key,
        "issued_on": invoice.issued_on,
        "supplier": invoice.supplier,
        "currency": invoice.currency,
        "line_items": [
            {"description": item.description, "quantity": item.quantity, "unit_minor": item.unit_minor}
            for item in invoice.items
        ],
        "subtotal_minor": subtotal,
        "tax_minor": invoice.tax_minor,
        "total_minor": total,
    }


def make_catalog() -> list[Invoice]:
    rng = random.Random(20261126)
    sources = {
        "train": (24, ("register", "memo", "checkout", "ledger")),
        "dev": (6, ("register", "memo", "checkout", "ledger")),
        "familiar": (8, ("register", "memo", "checkout", "ledger")),
        "novel": (8, ("packing", "remittance", "service", "notice")),
        "independent": (4, ("separate",)),
        "release": (10, ("packing", "remittance", "service", "notice", "separate")),
    }
    locales = (("en-US", "USD"), ("en-GB", "GBP"), ("de-DE", "EUR"), ("fr-FR", "EUR"), ("en-US", "CAD"))
    services = ("Cooling filter", "Cable sleeve", "Bench calibration", "Door seal", "Grounding clip",
                "Rack inspection", "Meter label", "Packing insert", "Carrier adjustment", "Mounting plate")
    catalog = []
    seen_keys = set()
    supplier_codes = rng.sample(range(1000, 9999), sum(n for n, _ in sources.values()))
    supplier_index = 0
    for split, (family_count, layouts) in sources.items():
        for family_number in range(family_count):
            supplier_code = supplier_codes[supplier_index]
            supplier_index += 1
            layout = layouts[family_number % len(layouts)]
            family = f"{split}-{family_number:02d}"
            locale, currency = locales[family_number % len(locales)]
            for row in range(SPLIT_SIZES[split] // family_count):
                code = f"SF-{rng.randrange(100000, 1000000)}"
                while code in seen_keys:
                    code = f"SF-{rng.randrange(100000, 1000000)}"
                seen_keys.add(code)
                items = tuple(
                    Line(kind, rng.randint(1, 9), rng.randrange(200, 24800))
                    for kind in rng.sample(services, rng.randrange(1, 4))
                )
                subtotal = sum(x.quantity * x.unit_minor for x in items)
                tax = (subtotal * (7 if currency == "USD" else 13) + 50) // 100
                if split in {"novel", "independent", "release"} and row % 12 == 4:
                    tax = None
                displayed = subtotal + tax + 101 if tax is not None and row % 12 == 7 else None
                issued_on = (date(2025, 1, 1) + timedelta(days=rng.randint(0, 590))).isoformat()
                catalog.append(Invoice(code, family, f"Sample Supplier {supplier_code}", issued_on,
                                       currency, items, tax, locale, split, layout, displayed))
    return catalog


def formatted_money(minor: int | None, locale: str) -> str:
    if minor is None:
        return "-"
    dollars, cents = divmod(minor, 100)
    number = f"{dollars:,}.{cents:02d}"
    if locale == "de-DE":
        return number.replace(",", "_").replace(".", ",").replace("_", ".")
    if locale == "fr-FR":
        return number.replace(",", " ").replace(".", ",")
    return number


def invoice_text(invoice: Invoice) -> str:
    gold = invoice_record(invoice)
    money = lambda value: formatted_money(value, invoice.locale)
    identifier = f"{invoice.key}"
    supplier = invoice.supplier
    issued = invoice.issued_on
    currency = invoice.currency
    subtotal = money(gold["subtotal_minor"])
    tax = money(gold["tax_minor"])
    total = money(gold["total_minor"])
    register = [f"{row.description} | {row.quantity} | {money(row.unit_minor)}" for row in invoice.items]
    inline = [f"{row.quantity} x {row.description} @ {money(row.unit_minor)} per unit" for row in invoice.items]
    service = [f"{row.description}: {money(row.unit_minor)} each for {row.quantity} units" for row in invoice.items]
    if invoice.layout == "register":
        return "\n".join([f"Supplier: {supplier}", f"Invoice: {identifier}", f"Issued: {issued}", f"Currency: {currency}",
                          "Description | Quantity | Unit price", *register,
                          f"Subtotal: {subtotal}", f"Tax: {tax}", f"Amount due: {total}"])
    if invoice.layout == "memo":
        return "\n".join([f"{supplier} billing memo / {issued}", f"Account invoice {identifier} [{currency}]",
                          *inline, f"Total due {total} including tax {tax}; net amount {subtotal}."])
    if invoice.layout == "checkout":
        return "\n".join([f"{identifier} / supplier {supplier}", f"Date of issue {issued} / All amounts {currency}",
                          *service, f"Net {subtotal}", f"Sales tax {tax}", f"Payable {total}"])
    if invoice.layout == "ledger":
        return "\n".join([f"{supplier} | issued {issued} | invoice {identifier}", f"Ledger in {currency}",
                          "Service | Count | Per unit", *register, f"Before tax {subtotal}",
                          f"Tax charged {tax}", f"Balance {total}"])
    if invoice.layout == "packing":
        return "\n".join([f"Billing details / {identifier}", f"Sold by {supplier}", f"Document date {issued}",
                          f"Invoice currency {currency}", f"Balance payable {total}", *service,
                          f"Merchandise {subtotal}", f"Tax {tax}"])
    if invoice.layout == "remittance":
        return "\n".join([f"Remittance advice for {identifier} ({currency})", f"{supplier} / Document issued {issued}",
                          f"Pay this amount {total}", *inline, f"Net charges {subtotal} / Tax {tax}"])
    if invoice.layout == "service":
        return "\n".join([f"Ref {identifier} / {currency}", f"Service fee notice, {issued}", supplier,
                          *[f"Unit {money(row.unit_minor)} / {row.quantity} times / {row.description}" for row in invoice.items],
                          f"Fee subtotal {subtotal}", f"Plus tax {tax}", f"Amount requested {total}"])
    if invoice.layout == "notice":
        return "\n".join([f"{supplier} / statement {identifier}", f"Issued on {issued}",
                          f"Invoice denomination {currency}", "One-off charges", *service,
                          f"Pre-tax {subtotal} / Tax assessment {tax} / Final bill {total}"])
    return "\n".join([f"Document {identifier}", f"Biller {supplier} / As of {issued} / {currency}", *inline,
                      f"Gross {total}", f"Breakdown: net {subtotal} and taxes {tax}."])


def _strict_dict(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key {key!r}")
        result[key] = value
    return result


def parse_record(completion: str | dict[str, Any]) -> dict[str, Any] | None:
    if isinstance(completion, str):
        try:
            data = json.loads(completion, object_pairs_hook=_strict_dict)
        except (ValueError, TypeError, RecursionError):
            return None
    else:
        data = completion
    keys = {"invoice_id", "issued_on", "supplier", "currency", "line_items", "subtotal_minor", "tax_minor", "total_minor"}
    if not isinstance(data, dict) or set(data) != keys:
        return None
    for name in ("invoice_id", "supplier", "issued_on", "currency"):
        value = data[name]
        if value is not None and (not isinstance(value, str) or not value.strip()):
            return None
    try:
        if data["issued_on"] is not None and date.fromisoformat(data["issued_on"]).isoformat() != data["issued_on"]:
            return None
    except ValueError:
        return None
    if data["currency"] is not None and re.fullmatch("[A-Z]{3}", data["currency"]) is None:
        return None
    for name in ("subtotal_minor", "tax_minor", "total_minor"):
        if data[name] is not None and type(data[name]) is not int:
            return None
    if not isinstance(data["line_items"], list):
        return None
    for row in data["line_items"]:
        if not isinstance(row, dict) or set(row) != {"description", "quantity", "unit_minor"}:
            return None
        if type(row["description"]) is not str or type(row["quantity"]) is not int or type(row["unit_minor"]) is not int:
            return None
        if not row["description"].strip() or row["quantity"] <= 0:
            return None
    return data


def score_record(completion: str | dict[str, Any], invoice: Invoice) -> dict[str, Any]:
    truth = invoice_record(invoice)
    observed = parse_record(completion)
    field_correct = {key: observed is not None and observed.get(key) == value
                     for key, value in truth.items() if key != "line_items"}
    keys = ("description", "quantity", "unit_minor")
    expected = Counter(tuple(r[k] for k in keys) for r in truth["line_items"])
    predicted = Counter(tuple(r[k] for k in keys) for r in observed["line_items"]) if observed is not None else Counter()
    matches = (expected & predicted).total()
    row_count = expected.total() + predicted.total()
    return {"valid": observed is not None, "exact": observed == truth,
            "fields": sum(field_correct.values()) / len(field_correct), "field_correct": field_correct,
            "row_f1": 2 * matches / row_count if row_count else 1.0}


def rules_extract(text: str) -> dict[str, Any] | None:
    lines = text.splitlines()
    if len(lines) < 9 or lines[4] != "Description | Quantity | Unit price":
        return None
    property_names = ("supplier", "invoice_id", "issued_on", "currency")
    headers = ("Supplier: ", "Invoice: ", "Issued: ", "Currency: ")
    if any(not line.startswith(header) for line, header in zip(lines, headers)):
        return None
    data = {name: line.removeprefix(header) for name, line, header in zip(property_names, lines, headers)}

    def minor(text_amount: str) -> int | None:
        if text_amount == "-":
            return None
        if re.fullmatch(r"(?:\d+|\d{1,3}(?:[., ]\d{3})+)[.,]\d{2}", text_amount) is None:
            raise ValueError("Unsupported amount format")
        units, frac = text_amount[:-3], text_amount[-2:]
        return int(re.sub(r"\D", "", units)) * 100 + int(frac)

    try:
        rows = [line.split(" | ") for line in lines[5:-3]]
        data["line_items"] = [{"description": desc, "quantity": int(qty), "unit_minor": minor(unit)}
                              for desc, qty, unit in rows]
        data["subtotal_minor"] = minor(lines[-3].removeprefix("Subtotal: "))
        data["tax_minor"] = minor(lines[-2].removeprefix("Tax: "))
        data["total_minor"] = minor(lines[-1].removeprefix("Amount due: "))
    except ValueError:
        return None
    return parse_record(data)


def examples_for(split: str, limit: int | None = None) -> list[Invoice]:
    found = [invoice for invoice in make_catalog() if invoice.split == split]
    if not found:
        raise ValueError(f"Unknown invoice split {split!r}")
    if limit is None or limit >= len(found):
        return found
    if limit <= 0:
        raise ValueError("Limit must be positive")
    groups: dict[str, list[Invoice]] = {}
    for invoice in found:
        groups.setdefault(invoice.family, []).append(invoice)
    balanced = [invoice for group_rows in zip(*groups.values(), strict=True) for invoice in group_rows]
    return balanced[:limit]

In [3]:
"""Original invoice page and chart rendering for the document notebooks."""

from PIL import Image, ImageDraw, ImageFont

# The notebooks inline this file after the fixture contract. The fallback import
# keeps the same functions usable in ordinary Python.
if "invoice_text" not in globals():
    from invoice_fixtures import Invoice, invoice_record, invoice_text


def _font(size: int, bold: bool = False):
    name = "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"
    try:
        return ImageFont.truetype(name, size)
    except OSError:
        return ImageFont.load_default(size=size)


def invoice_image(invoice: Invoice) -> Image.Image:
    image = Image.new("RGB", (896, 624), "#faf9f5")
    draw = ImageDraw.Draw(image)
    fields = invoice_text(invoice).splitlines()
    accents = {
        "register": "#245067", "memo": "#732f41", "checkout": "#30614e", "ledger": "#2a3e72",
        "packing": "#31535c", "remittance": "#694323", "service": "#4b3f65", "notice": "#73422d",
    }
    accent = accents.get(invoice.layout, "#4a5744")
    draw.rounded_rectangle((32, 29, 864, 595), 11, fill="white", outline="#d9dad8", width=2)
    draw.rectangle((33, 34, 863, 42), fill=accent)
    draw.text((64, 63), "Supplier billing", font=_font(28, bold=True), fill=accent)
    draw.text((64, 102), "Original synthetic course fixture", font=_font(15), fill="#5b6462")
    draw.line((61, 134, 835, 134), fill="#d5dad8", width=2)
    for i, line in enumerate(fields):
        y = 153 + 35 * i
        if i % 2:
            draw.rectangle((58, y - 3, 837, y + 28), fill="#f5f7f6")
        is_heading = line.startswith(("Description |", "Service |", "One-off charges"))
        draw.text((65, y), line, fill="#233034", font=_font(20, bold=is_heading))
        if draw.textbbox((65, y), line, font=_font(20, bold=is_heading))[2] >= 837:
            raise ValueError(f"Invoice field did not fit the page: {line!r}")
    return image


def spend_chart(invoice: Invoice, *, varied: bool = False):
    """Render line-item spending as labelled bars with separately known cents."""
    gold = invoice_record(invoice)
    values = [row["quantity"] * row["unit_minor"] for row in gold["line_items"]]
    image = Image.new("RGB", (896, 512), "#ffffff")
    draw = ImageDraw.Draw(image)
    accent = "#266978" if varied else "#27598b"
    order = list(reversed(range(len(values)))) if varied else list(range(len(values)))
    draw.text((48, 39), f"Line item spending · {invoice.currency}", fill=accent, font=_font(28, bold=True))
    draw.text((48, 80), "Total before tax per line item", fill="#53575b", font=_font(17))
    largest = max(values)
    rows = []
    for position, index in enumerate(order):
        row = gold["line_items"][index]
        value = values[index]
        y = 141 + 86 * position
        draw.text((51, y), row["description"], font=_font(19), fill="#222222")
        money = f"{value / 100:,.2f}"
        draw.text((734, y), money, font=_font(19, bold=True), fill="#222222")
        draw.rounded_rectangle((51, y + 34, 51 + int(value / largest * 650), y + 55), 4, fill=accent)
        rows.append({"description": row["description"], "line_total_minor": value})
    return image, {"currency": invoice.currency, "rows": rows}

## Which tokens receive a loss?

Both renders use the same image tokens. The prompt prefix must match the full rendered sequence exactly before the mask removes it.

In [4]:
"""Bounded Qwen2-VL training and evaluation shared by the document notebooks."""

from dataclasses import dataclass
from pathlib import Path
import random
from time import perf_counter

import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration

VISION_MODEL = "Qwen/Qwen2-VL-2B-Instruct"
VISION_REV = "1860c46f12b7449d0f5b426a7474ab715342bc65"
VISION_SEED = 42
VISION_MAX_PIXELS = 576 * 28 * 28
VISION_MAX_OUTPUT = 220


@dataclass
class DocumentExample:
    key: str
    image: object
    instruction: str
    answer: str
    group: str


def processor_for_vision():
    return AutoProcessor.from_pretrained(
        VISION_MODEL, revision=VISION_REV, trust_remote_code=False,
        min_pixels=4 * 28 * 28, max_pixels=VISION_MAX_PIXELS,
    )


def load_vision_base(trainable: bool = False):
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime. This vision lab uses CUDA.")
    quantisation = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    base = Qwen2VLForConditionalGeneration.from_pretrained(
        VISION_MODEL, revision=VISION_REV, trust_remote_code=False, use_safetensors=True,
        torch_dtype=torch.float16, device_map={"": 0}, quantization_config=quantisation,
        attn_implementation="sdpa",
    )
    if trainable:
        return prepare_model_for_kbit_training(base, gradient_checkpointing_kwargs={"use_reentrant": False})
    return base.eval()


def adapter_for_vision(base):
    adapter = get_peft_model(
        base, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
                         lora_dropout=0.0, task_type="CAUSAL_LM"),
    )
    trainable = [name for name, value in adapter.named_parameters() if value.requires_grad]
    if not trainable or any(".visual." in name or "lora" not in name for name in trainable):
        raise RuntimeError("Expected language-backbone LoRA parameters only.")
    return adapter


def encode_image(processor, item: DocumentExample, assistant: str | None = None):
    chat = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": item.instruction}]}]
    if assistant is not None:
        chat.append({"role": "assistant", "content": [{"type": "text", "text": assistant}]})
    template = processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=assistant is None)
    # Use the exact same pixels and processor settings in the prompt and full sequence.
    inputs = processor(text=[template], images=[item.image], padding=False, return_tensors="pt")
    return inputs.to("cuda")


def sft_feature(processor, item: DocumentExample):
    prompt = encode_image(processor, item)
    full = encode_image(processor, item, item.answer)
    labels = full["input_ids"].clone()
    prefix = prompt["input_ids"]
    width = prefix.shape[1]
    if width >= labels.shape[1] or not torch.equal(labels[:, :width], prefix):
        raise ValueError(f"Chat template changed its expanded image prefix for {item.key}.")
    labels[:, :width] = -100
    if not torch.any(labels[:, width:] != -100):
        raise ValueError("The prompt mask removed the entire target.")
    return full, labels


def fit_vision(adapter, processor, items, *, updates: int, accumulation: int = 2):
    random.seed(VISION_SEED)
    order = list(items)
    random.shuffle(order)
    if updates <= 0 or not order:
        raise ValueError("Vision training needs a positive update budget and at least one example.")
    adapter.train()
    optimiser = torch.optim.AdamW((p for p in adapter.parameters() if p.requires_grad), lr=1e-4)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = perf_counter()
    history = []
    for update in range(updates):
        optimiser.zero_grad(set_to_none=True)
        step_loss = 0.0
        for j in range(accumulation):
            features, labels = sft_feature(processor, order[(update * accumulation + j) % len(order)])
            scalar = adapter(**features, labels=labels, use_cache=False).loss
            if not torch.isfinite(scalar):
                raise RuntimeError(f"Non-finite vision SFT loss at step {update}.")
            (scalar / accumulation).backward()
            step_loss += scalar.detach().item() / accumulation
        norm = torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        if not torch.isfinite(norm):
            raise RuntimeError("Non-finite adapter gradient.")
        optimiser.step()
        history.append(step_loss)
    torch.cuda.synchronize()
    adapter.eval()
    return {"updates": updates, "examples_seen": updates * accumulation, "seconds": perf_counter() - started,
            "peak_gib": torch.cuda.max_memory_allocated() / 2**30, "loss": history}


@torch.inference_mode()
def generate_vision(model, processor, example, *, max_new_tokens=VISION_MAX_OUTPUT, **options):
    inputs = encode_image(processor, example)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, **options)
    ids = output[0, inputs["input_ids"].shape[1]:]
    value = processor.tokenizer.decode(ids, skip_special_tokens=True).strip()
    stops = model.generation_config.eos_token_id
    eos = stops if isinstance(stops, (list, tuple)) else [stops]
    return {"text": value, "tokens": ids.cpu().tolist(), "complete": bool(len(ids) and ids[-1].item() in eos)}


def save_vision_adapter(model, path: str) -> Path:
    target = Path(path)
    model.save_pretrained(target, safe_serialization=True)
    if not (target / "adapter_model.safetensors").is_file():
        raise RuntimeError(f"Missing adapter weights at {target}")
    return target

In [5]:
import json
from dataclasses import asdict
from importlib.metadata import version
from pathlib import Path
from time import perf_counter
import hashlib
import random
import shutil

if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("This notebook needs a Colab T4 runtime.")
torch.manual_seed(VISION_SEED)
START = perf_counter()
print("GPU", torch.cuda.get_device_name(0))
for package in ("torch", "transformers", "peft", "bitsandbytes", "Pillow"):
    print(package, version(package))

def paired_family_exact(before, after, get_exact, seed=42):
    baseline = {row["key"]: row for row in before}
    assert len(baseline) == len(before) == len(after)
    assert set(baseline) == {row["key"] for row in after}
    deltas = {}
    for current in after:
        previous = baseline[current["key"]]
        assert previous["family"] == current["family"]
        deltas.setdefault(current["family"], []).append(get_exact(current) - get_exact(previous))
    families = sorted(deltas)
    if len(families) < 2:
        return {"families":len(families),"interval_pp":None}
    rng = random.Random(seed)
    draws = []
    for _ in range(2000):
        rows = [delta for family in rng.choices(families, k=len(families)) for delta in deltas[family]]
        draws.append(100 * sum(rows) / len(rows))
    draws.sort()
    return {"families":len(families),
            "delta_pp":100 * sum(sum(values) for values in deltas.values()) / len(after),
            "interval_pp":[draws[49],draws[1949]]}

def feature_manifest(items, updates, accumulation):
    return {
        "model":VISION_MODEL, "revision":VISION_REV, "catalogue":CATALOG_VERSION,
        "updates":updates, "accumulation":accumulation, "seed":VISION_SEED,
        "features_sha256":hashlib.sha256(json.dumps([
            [item.key, item.group, item.instruction, item.answer,
             hashlib.sha256(item.image.tobytes()).hexdigest()] for item in items
        ],ensure_ascii=False).encode()).hexdigest(),
    }

def write_results(filename, payload):
    model = {"id": VISION_MODEL, "revision": VISION_REV}
    versions = {p: version(p) for p in ("torch","transformers","peft","bitsandbytes","Pillow","datasets")}
    experiment = {"elapsed_seconds": round(perf_counter()-START, 3),
                  "gpu": torch.cuda.get_device_name(0), "model": model,
                  "packages": versions, "catalogue": CATALOG_VERSION, "seed": VISION_SEED}
    with open(filename, "w", encoding="utf-8") as handle:
        json.dump({**experiment, **payload}, handle, indent=2, ensure_ascii=False)
    print(json.dumps({**experiment, **payload}, ensure_ascii=False))

GPU Tesla T4
torch 2.14.1+cu130
transformers 4.51.3
peft 0.15.2
bitsandbytes 0.50.2
Pillow 11.3.0


## What would warrant a release?

Freeze quality gates and output contracts before training. Count missing fields and rows, unparseable responses, and premature termination. Use an independent external invoice benchmark for header fields only.

In [6]:
GATES={"valid":0.98,"complete":0.98,"exact_complete":0.90,"slice_regression_pp":3.0}
TASK = VISIBLE_CONTRACT
TRAIN_UPDATES=32
RL_GROUPS=8
ROLLOUTS=3
MAX_TOKENS=220
SEED=42
EXTERNAL_REV="446bab08662790cb39457792243bff5704373b95"

def cap_item(row):
    return DocumentExample(row.key,invoice_image(row),TASK,json.dumps(invoice_record(row),ensure_ascii=False),row.family)

def cap_chart_item(row):
    image,target=spend_chart(row)
    prompt="Read the bar chart. Return exactly one JSON object with currency and rows. Each row must contain description and line_total_minor in cents. Preserve visual row order."
    return DocumentExample(row.key,image,prompt,json.dumps(target,ensure_ascii=False),row.family)

training=[cap_item(row) for row in examples_for("train",72)]+[
    cap_chart_item(row) for row in examples_for("train",24)]
rollouts=examples_for("dev",RL_GROUPS)
benchmark=examples_for("release",84)
assert len({row.family for row in benchmark})==10
assert sum(row.locale in {"de-DE","fr-FR"} for row in benchmark) >= 30
assert sum(row.locale not in {"de-DE","fr-FR"} for row in benchmark) >= 30
assert {item.group for item in training}.isdisjoint(row.family for row in rollouts+benchmark)
test_hash=hashlib.sha256("\n".join(row.key for row in benchmark).encode()).hexdigest()
print("Release gates:",GATES,"cases:",len(benchmark),"release split SHA256:",test_hash)

Release gates: {'valid': 0.98, 'complete': 0.98, 'exact_complete': 0.9, 'slice_regression_pp': 3.0} cases: 84 release split SHA256: bd16e86b4b9ed1e63c4c3e93cca9e51bdde6746f7a04b19708471f23313614b1


## Which Lab 08 artefact is the incumbent?

Verify the checked notebook and adapter files before spending T4 time. The paired course run requires the exported Lab 08 adapter. Explicit standalone runs record a separate refit.

In [7]:
"""Read a passed Lesson 08 export and verify the capstone's input contract."""

import hashlib
import json
import os
from pathlib import Path
import re


def sha256_file(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def read_prior_handoff(expected_features, output_root=None, explicit_dir=None):
    root = Path(output_root or Path.cwd().parent).resolve()
    explicit = explicit_dir or os.environ.get("LAB08_HANDOFF_DIR")
    if explicit:
        candidates = [Path(explicit).resolve()]
    else:
        candidates = [
            folder for folder in root.glob("08-extract-tables-and-charts-*")
            if (folder / "receipt.json").is_file()
            and json.loads((folder / "receipt.json").read_text())["status"] == "passed"
        ]
    if not candidates:
        return None
    if len(candidates) != 1:
        raise ValueError("Set LAB08_HANDOFF_DIR to select the approved Lab 08 job.")
    folder = candidates[0]
    receipt = json.loads((folder / "receipt.json").read_text())
    source = "08-extract-tables-and-charts.ipynb"
    if receipt.get("status") != "passed" or receipt.get("return_code") != 0 or receipt.get("source") != source:
        raise ValueError("The Lab 08 receipt must name a successful run of the structure notebook.")
    if receipt["source_sha256"] != sha256_file(folder / source):
        raise ValueError("Lab 08 source notebook does not match its receipt.")
    if receipt["executed_sha256"] != sha256_file(folder / "executed.ipynb"):
        raise ValueError("Lab 08 executed notebook does not match its receipt.")
    payload = json.loads((folder / "08_handoff.json").read_text())
    if payload["features"] != expected_features:
        raise ValueError("The Lab 08 feature recipe differs from the capstone recipe.")
    weights = folder / "08_structure_adapter"
    adapter_config = json.loads((weights / "adapter_config.json").read_text())
    if (adapter_config.get("base_model_name_or_path") != expected_features["model"]
            or adapter_config.get("r") != 8 or adapter_config.get("lora_alpha") != 16
            or set(adapter_config.get("target_modules", [])) != {"q_proj", "v_proj"}):
        raise ValueError("The Lab 08 adapter configuration differs from the capstone recipe.")
    digest = sha256_file(weights / "adapter_model.safetensors")
    if payload["adapter_sha256"] != digest:
        raise ValueError("Lab 08 adapter bytes differ from the handoff.")
    if (not re.fullmatch(r"SF-\d+", payload["parity_key"])
            or not payload["parity_tokens"]
            or any(type(token) is not int or token < 0 for token in payload["parity_tokens"])):
        raise ValueError("The Lab 08 handoff has no valid greedy-token parity case.")
    return weights, payload, receipt


def require_token_parity(item, saved_key, saved_tokens, observed_tokens):
    if item.key != saved_key or list(observed_tokens) != list(saved_tokens):
        raise ValueError("The restored adapter changed the fixed-case greedy tokens.")

pins = {"transformers": "4.51.3", "peft": "0.15.2", "accelerate": "1.6.0",
        "bitsandbytes": "0.50.2", "pillow": "11.3.0", "pypdfium2": "4.30.0"}
for package, wanted in pins.items():
    if version(package) != wanted:
        raise RuntimeError(f"Expected {package}=={wanted}, got {version(package)}")
get_ipython().run_line_magic("config", "InlineBackend.figure_formats = ['svg']")
expected_features=feature_manifest(training,TRAIN_UPDATES,2)
prior_handoff=read_prior_handoff(expected_features)
if not prior_handoff and os.environ.get("LAB15_STANDALONE_REFIT") != "1":
    raise RuntimeError("Stage the verified Lab 08 export or explicitly enable standalone refit.")
assert len({item.key for item in benchmark}) == len(benchmark)
assert {item.key for item in training}.isdisjoint(row.key for row in rollouts+benchmark)
assert {row.family for row in rollouts}.isdisjoint(row.family for row in benchmark)

## Which outputs fool a usable-looking verifier?

The training reward prefers schema-valid responses whose row content, currency and totals match the printed values. The offline release guard checks currency and row identities using an independent verified text view of the same fixture. A production document with only a page image would require its own trusted text path before using this guard.

In [8]:
from collections import Counter
from copy import deepcopy
from gc import collect

SOURCE_ROW = re.compile(
    r"^(?:([A-Za-z ]+): .* each for [0-9]+ units|"
    r"[0-9]+ x ([A-Za-z ]+) @ .* per unit|"
    r"Unit .* / [0-9]+ times / ([A-Za-z ]+))$",
    re.MULTILINE,
)

def verified_source_guard(text, source_text):
    value=parse_record(text)
    if value is None or not all(value[name] is not None for name in ("invoice_id","currency","supplier")):
        return False
    currencies=set(re.findall(r"\b(?:USD|GBP|EUR|CAD)\b",source_text))
    expected=Counter(next(part for part in match.groups() if part is not None)
                     for match in SOURCE_ROW.finditer(source_text))
    observed=Counter(item["description"] for item in value["line_items"])
    return currencies=={value["currency"]} and bool(expected) and observed==expected

def training_reward(text, expected, complete):
    observed=parse_record(text)
    if observed is None:
        return -0.5
    scalar_fields=("invoice_id","issued_on","supplier","currency","subtotal_minor","tax_minor","total_minor")
    field_score=sum(observed[k]==expected[k] for k in scalar_fields)/len(scalar_fields)
    keys=("description","quantity","unit_minor")
    rows=lambda r:Counter(tuple(item[k] for k in keys) for item in r["line_items"])
    predicted,target=rows(observed),rows(expected)
    f1=2*(predicted&target).total()/max(1,predicted.total()+target.total())
    return 0.30*field_score+0.55*f1+0.15*int(complete)

control_invoice=next(row for row in benchmark if len(row.items)>=2)
control_source=invoice_text(control_invoice)
check=invoice_record(control_invoice)
wrong_currency=deepcopy(check)
wrong_currency["currency"]="GBP" if check["currency"]=="USD" else "USD"
missing_row=deepcopy(check)
missing_row["line_items"].pop()
assert training_reward(json.dumps(check),check,True)==1
assert training_reward(json.dumps(wrong_currency),check,True)<1
assert training_reward(json.dumps(missing_row),check,True)<1
assert training_reward(json.dumps(check)[:-1],check,False)<0
controls={"grounded_record_accepted":verified_source_guard(json.dumps(check),control_source),
          "wrong_currency_rejected":not verified_source_guard(json.dumps(wrong_currency),control_source),
          "missing_row_rejected":not verified_source_guard(json.dumps(missing_row),control_source),
          "truncated_record_rejected":not verified_source_guard(json.dumps(check)[:-1],control_source)}
assert all(controls.values()), controls
print("Verified source controls:",controls)

Verified source controls: {'grounded_record_accepted': True, 'wrong_currency_rejected': True, 'missing_row_rejected': True, 'truncated_record_rejected': True}


## Does one full-width policy update fit on the T4?

Run one real update on a development invoice with three sampled completions forced to the generation cap. Diagnostic advantages guarantee backward passes. These diagnostic weights and rewards never enter the scored comparison.

In [9]:
def rollout_for(model, processor, item, completion_width, temperature, force_width=False):
    inputs=encode_image(processor,item)
    model.eval()
    with torch.no_grad():
        result=model.generate(**inputs,max_new_tokens=completion_width,do_sample=True,
                              temperature=temperature,top_k=0,top_p=1.0,
                              min_new_tokens=completion_width if force_width else 0)
    generated=result[:,inputs["input_ids"].shape[1]:].clone()
    eos=model.generation_config.eos_token_id
    stops=eos if isinstance(eos,(tuple,list)) else [eos]
    done=bool(generated.numel() and generated[0,-1].item() in stops)
    return generated,processor.tokenizer.decode(generated[0],skip_special_tokens=True),done

def sampled_logs(model,inputs,tokens,temperature):
    n=inputs["input_ids"].shape[1]
    ids=torch.cat((inputs["input_ids"],tokens),dim=1)
    kwargs={key:value for key,value in inputs.items() if key not in {"input_ids","attention_mask"}}
    output=model(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False,**kwargs)
    logits=output.logits[:,n-1:-1,:].float()/temperature
    return torch.log_softmax(logits,dim=-1).gather(-1,tokens[...,None]).squeeze(-1)

def optimise_with_rewards(model,processor,training_invoices, *, profile_only=False):
    optimiser=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=1e-5)
    history=[]
    torch.manual_seed(SEED)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started=perf_counter()
    for row in training_invoices:
        item=cap_item(row)
        inputs=encode_image(processor,item)
        generated=[rollout_for(model,processor,item,MAX_TOKENS,0.75,force_width=profile_only) for _ in range(ROLLOUTS)]
        generated=[run for run in generated if run[0].numel()]
        rewards=[training_reward(text,invoice_record(row),done) for _,text,done in generated]
        if profile_only and any(run[0].shape[1] != MAX_TOKENS for run in generated):
            raise RuntimeError("Profile completions did not reach the fixed token width.")
        profile_rewards=[-1.0,0.0,1.0] if profile_only else rewards
        values=torch.tensor(profile_rewards,dtype=torch.float32)
        spread=values.std(unbiased=False).item() if len(rewards)>1 else 0
        losses=[]
        if spread>1e-8:
            normalised=((values-values.mean())/(spread+1e-6)).tolist()
            model.eval()
            with torch.no_grad():
                old=[sampled_logs(model,inputs,tokens,0.75).detach() for tokens,_,_ in generated]
            model.train()
            for _ in range(1 if profile_only else 2):
                optimiser.zero_grad(set_to_none=True)
                group_loss=0.0
                for (tokens,_,_),previous,advantage in zip(generated,old,normalised,strict=True):
                    ratio=(sampled_logs(model,inputs,tokens,0.75)-previous).exp()
                    loss=-torch.minimum(ratio*advantage,ratio.clamp(0.8,1.2)*advantage).mean()/len(old)
                    if not torch.isfinite(loss):
                        raise RuntimeError("Non-finite policy loss.")
                    loss.backward()
                    group_loss+=loss.detach().item()
                norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                if not torch.isfinite(norm):
                    raise RuntimeError("Non-finite reward gradient.")
                optimiser.step()
                losses.append(group_loss)
        history.append({"key":row.key,"rewards":rewards,"objective_reward_spread":spread,"token_lengths":[run[0].shape[1] for run in generated],"losses":losses})
    torch.cuda.synchronize()
    model.eval()
    return {"groups":history,"steps":sum(len(x["losses"]) for x in history),
            "elapsed_seconds":perf_counter()-started,"peak_gib":torch.cuda.max_memory_allocated()/2**30}


processor=processor_for_vision()
entry_free,_=torch.cuda.mem_get_info()
if entry_free < 12 * 2**30:
    raise RuntimeError("Start the capstone profile with at least 12 GiB free on the T4.")
probe=max(rollouts,key=lambda row:encode_image(processor,cap_item(row))["input_ids"].shape[1])
torch.cuda.empty_cache()
start_allocated=torch.cuda.memory_allocated()
torch.manual_seed(SEED)
probe_model=load_vision_base(trainable=True)
if prior_handoff:
    probe_model=PeftModel.from_pretrained(probe_model,prior_handoff[0],is_trainable=True)
else:
    probe_model=adapter_for_vision(probe_model)
profile=optimise_with_rewards(probe_model,processor,[probe],profile_only=True)
profile["entry_free_gib"]=entry_free/2**30
profile["peak_reserved_gib"]=torch.cuda.max_memory_reserved()/2**30
del probe_model
collect()
torch.cuda.empty_cache()
profile["retained_bytes"]=max(0,torch.cuda.memory_allocated()-start_allocated)
Path("15_t4_profile.json").write_text(json.dumps(profile,indent=2),encoding="utf-8")
print(json.dumps(profile,indent=2))
if (profile["steps"] != 1 or profile["elapsed_seconds"] > 90 or profile["peak_gib"] > 10
        or profile["peak_reserved_gib"] > 12 or profile["retained_bytes"] > 128*2**20
        or profile["peak_reserved_gib"]*2**30 > entry_free-2*2**30):
    raise RuntimeError("The one-update capstone profile exceeded the T4 limits.")
torch.manual_seed(SEED)

{
  "groups": [
    {
      "key": "SF-252566",
      "rewards": [
        -0.5,
        -0.5,
        -0.5
      ],
      "objective_reward_spread": 0.8164966106414795,
      "token_lengths": [
        220,
        220,
        220
      ],
      "losses": [
        0.0
      ]
    }
  ],
  "steps": 1,
  "elapsed_seconds": 68.76289728499978,
  "peak_gib": 2.787752628326416,
  "entry_free_gib": 14.46075439453125,
  "peak_reserved_gib": 3.28125,
  "retained_bytes": 17040384
}


## How good are the starting points?

The rules see verified text. The model sees the rendered image. Evaluations use the same release rows and deterministic decoding.

In [10]:
def mean_field(entries,key):
    return sum(entry[key] for entry in entries)/len(entries)

def evaluate_release(model):
    scored=[]
    for row in benchmark:
        response=generate_vision(model,processor,cap_item(row),max_new_tokens=MAX_TOKENS)
        raw=score_record(response["text"],row)
        scored.append({"key":row.key,"family":row.family,"layout":row.layout,"locale":row.locale,
                       "complete":response["complete"],"guard":verified_source_guard(response["text"],invoice_text(row)),
                       "completion":response["text"],**raw})
    return scored

def compact(entries):
    metrics={metric:mean_field(entries,metric) for metric in ("valid","exact","fields","row_f1","complete")}
    metrics["exact_complete"]=sum(row["exact"] and row["complete"] for row in entries)/len(entries)
    return metrics

baseline_model=load_vision_base()
baseline=evaluate_release(baseline_model)
rules=[score_record(rules_extract(invoice_text(row)),row) for row in benchmark]
print("Rules exact:",mean_field(rules,"exact"))
print("Frozen VLM:",compact(baseline))
del baseline_model
collect()
torch.cuda.empty_cache()

Rules exact: 0.0
Frozen VLM: {'valid': 0.0, 'exact': 0.0, 'fields': 0.0, 'row_f1': 0.0, 'complete': 0.9880952380952381, 'exact_complete': 0.0}


## Can we recover the supervised incumbent?

Use the completed lesson 8 adapter when the queue has its manifest. Check the features, adapter digest and fixed anchor output before training on rewards. A standalone run can rebuild the same recipe and records that path.

In [11]:
if prior_handoff:
    weights,handoff,source_receipt=prior_handoff
    model=PeftModel.from_pretrained(load_vision_base(trainable=True),weights,is_trainable=True).eval()
    anchor=cap_item(examples_for("familiar",1)[0])
    require_token_parity(anchor,handoff["parity_key"],handoff["parity_tokens"],
                         generate_vision(model,processor,anchor)["tokens"])
    sft_record={"path":"lesson_08","feature_manifest":expected_features,
                "adapter_sha256":handoff["adapter_sha256"],
                "source_sha256":source_receipt["source_sha256"],
                "executed_sha256":source_receipt["executed_sha256"],"anchor_token_parity":True}
else:
    model=adapter_for_vision(load_vision_base(trainable=True))
    sft_record={"path":"standalone_refit","feature_manifest":expected_features,**fit_vision(
        model,processor,training,updates=TRAIN_UPDATES,accumulation=2)}
sft=evaluate_release(model)
incumbent=save_vision_adapter(model,"15_incumbent_adapter")
incumbent_sha=hashlib.sha256((incumbent/"adapter_model.safetensors").read_bytes()).hexdigest()
fixture=cap_item(benchmark[0])
incumbent_tokens=generate_vision(model,processor,fixture,max_new_tokens=MAX_TOKENS)["tokens"]
serving_config={"model":VISION_MODEL,"revision":VISION_REV,"adapter_sha256":incumbent_sha,
                "max_new_tokens":MAX_TOKENS,"decoding":"greedy","max_pixels":VISION_MAX_PIXELS}
Path("15_incumbent_serving.json").write_text(json.dumps(serving_config,indent=2),encoding="utf-8")
print("Supervised release set:",compact(sft),"adapter SHA256:",incumbent_sha)

Supervised release set: {'valid': 0.9880952380952381, 'exact': 0.0, 'fields': 0.03571428571428571, 'row_f1': 0.0, 'complete': 0.9880952380952381, 'exact_complete': 0.0} adapter SHA256: 592bbf3309b83409b446f8a0d5bcadff086a6d77eabd3f08a7b23c47565ad71c


## Does an image-based reward help?

Sample three completions per document. Score each independently, centre and scale the scores within a group, and skip groups with no reward variation. Re-evaluate token log probabilities under the frozen rollout policy and limit the ratio during two optimisation passes. This is a small reference-free GRPO comparison with a fixed generation budget.

In [12]:
rl_record=optimise_with_rewards(model,processor,rollouts)
rl=evaluate_release(model)
candidate=save_vision_adapter(model,"15_reward_adapter")
candidate_sha=hashlib.sha256((candidate/"adapter_model.safetensors").read_bytes()).hexdigest()
print("Reward updates:",rl_record["steps"],"candidate SHA256:",candidate_sha)
print("Reward-adapted release set:",compact(rl))

Reward updates: 10 candidate SHA256: 42243b0a47f29898667fab319aae438ba5c6813ea1e694ad1c7660cfa77141b5
Reward-adapted release set: {'valid': 0.9880952380952381, 'exact': 0.0, 'fields': 0.03571428571428571, 'row_f1': 0.0, 'complete': 0.9880952380952381, 'exact_complete': 0.0}


## How does the candidate handle an independent fixture source?

Read four pinned, single-page synthetic invoices from an external benchmark. Its labels describe totals and line amounts rather than our quantity and unit-price contract, so compare only the documented overlapping header fields. Keep this slice separate.

In [13]:
from decimal import Decimal
from huggingface_hub import hf_hub_download

import pypdfium2

EXTERNAL_DATA="drew-ipp/invoice-extraction-benchmark"
EXTERNAL_IDS=("clean-us-clean-simple","clean-de-grid-eu-decimals",
              "clean-fr-right-header","clean-gb-mono-plain")
def download_external(path):
    return Path(hf_hub_download(EXTERNAL_DATA,"corpus/v1/"+path,repo_type="dataset",revision=EXTERNAL_REV))

def external_headers(model):
    rows=[]
    for key in EXTERNAL_IDS:
        label=json.loads(download_external(f"answers/{key}.json").read_text())
        raw=download_external(label["file"]).read_bytes()
        if hashlib.sha256(raw).hexdigest()!=label["sha256"]:
            raise ValueError(f"Benchmark bytes differ for {key}.")
        pdf=pypdfium2.PdfDocument(raw)
        try:
            image=pdf[0].render(scale=1.25).to_pil().convert("RGB").copy()
        finally:
            pdf.close()
        output=generate_vision(model,processor,DocumentExample(key,image,TASK,"",key),max_new_tokens=MAX_TOKENS)
        gold=label["expected"]
        cents=lambda value:int((Decimal(str(value))*100).quantize(Decimal("1"))) if value is not None else None
        target={"invoice_id":gold["invoice_number"],"supplier":gold["vendor_name"],
                "currency":gold["currency"],"total_minor":cents(gold["total"])}
        observed=parse_record(output["text"])
        rows.append({"key":key,"valid":observed is not None,
                     "completion":output["text"],"gold":target,"fields":{name:observed is not None and observed.get(name)==truth for name,truth in target.items()}})
    return rows

external=external_headers(model)
print("Independent header checks:",external)

Independent header checks: [{'key': 'clean-us-clean-simple', 'valid': False, 'completion': '{"invoice_id": "123456", "issued_on": "2022-04-15", "supplier": "Acme Industrial", "currency": "USD", "line_items": [{"description": "Heavy-duty steel washers (pack of 200)", "quantity": 2, "unit_minor": 320000]}, {"description": "Threaslide adhesive, blue, 10ml", "quantity": 6, "unit_minor": 130000}], "subtotal_minor": 320000, "tax_minor": 640000, "total_minor": 960000}', 'gold': {'invoice_id': 'ACM-100145', 'supplier': 'Acme Industrial Supply Co', 'currency': 'USD', 'total_minor': 37249}, 'fields': {'invoice_id': False, 'supplier': False, 'currency': False, 'total_minor': False}}, {'key': 'clean-de-grid-eu-decimals', 'valid': True, 'completion': '{"invoice_id": "236", "issued_on": "2019-04-10", "supplier": "301 Hanover", "currency": "EUR", "line_items": [{"description": "Brake pads, trailer axle set", "quantity": 1, "unit_minor": 151}, {"description": "Hydraulic hose 1/2in, per metre", "quanti

## Will speculative decoding preserve the answer?

Benchmark ordinary greedy decoding and prompt lookup on interleaved runs with the same checkpoint. Refuse latency claims when the runtime cannot use the mode or when generated tokens differ.

In [14]:
import statistics
Q=benchmark[:6]
def latency_report(model):
    jobs=[cap_item(row) for row in Q]
    if rl_record["steps"]==0:
        disposition="no policy gradient signal"
    else:
        disposition="completed"
    try:
        generate_vision(model,processor,jobs[0],max_new_tokens=MAX_TOKENS)
        generate_vision(model,processor,jobs[0],max_new_tokens=MAX_TOKENS,prompt_lookup_num_tokens=4)
    except (AttributeError, TypeError, ValueError, NotImplementedError) as exc:
        return {"status":"unsupported","reason":str(exc),"training":disposition}
    trial=[]
    for index,item in enumerate(jobs):
        results={}
        settings=[("greedy",{}),("lookup",{"prompt_lookup_num_tokens":4})]
        if index%2:
            settings.reverse()
        for label,kwargs in settings:
            torch.cuda.synchronize()
            started=perf_counter()
            output=generate_vision(model,processor,item,max_new_tokens=MAX_TOKENS,**kwargs)
            torch.cuda.synchronize()
            results[label]={"seconds":perf_counter()-started,"tokens":output["tokens"],
                            "complete":output["complete"]}
        trial.append({"key":item.key,"matches":results["greedy"]["tokens"]==results["lookup"]["tokens"],"results":results})
    if not all(r["matches"] for r in trial):
        return {"status":"parity_failed","pairs":trial}
    if not all(out["complete"] for row in trial for out in row["results"].values()):
        return {"status":"incomplete_output","pairs":trial}
    durations={name: sorted(row["results"][name]["seconds"] for row in trial) for name in ("greedy","lookup")}
    return {"status":"parity_passed","cases":len(trial),"pairs":trial,
            "p50_seconds":{k:statistics.median(a) for k,a in durations.items()},
            "p95_nearest_rank_seconds":{k:a[-1] for k,a in durations.items()},
            "pages_per_gpu_second":{k:len(a)/sum(a) for k,a in durations.items()}}

serving=latency_report(model)
print("Serving status:",serving["status"])

Serving status: parity_passed


## Can the earlier parser be restored?

Release decisions use predeclared thresholds. If no group generated a useful reward gradient, record an inconclusive training result. Reload the saved supervised adapter from a fresh base and verify a documented request.

In [15]:
del model
collect()
torch.cuda.empty_cache()
restored=PeftModel.from_pretrained(load_vision_base(trainable=True),incumbent,is_trainable=False).eval()
restored_config=json.loads(Path("15_incumbent_serving.json").read_text())
assert restored_config["model"]==VISION_MODEL and restored_config["revision"]==VISION_REV
assert restored_config["max_pixels"]==VISION_MAX_PIXELS and restored_config["decoding"]=="greedy"
assert hashlib.sha256((incumbent/"adapter_model.safetensors").read_bytes()).hexdigest()==restored_config["adapter_sha256"]
restored_tokens=generate_vision(restored,processor,fixture,max_new_tokens=restored_config["max_new_tokens"])["tokens"]
require_token_parity(fixture,fixture.key,incumbent_tokens,restored_tokens)
scores=compact(rl)
incumbent_scores=compact(sft)
issues=[key for key,minimum in GATES.items() if key in scores and scores[key]<minimum]
slice_deltas={}
slice_checks={layout:(lambda r, layout=layout:r["layout"]==layout) for layout in sorted({r["layout"] for r in sft})}
slice_checks.update({"decimal_comma":lambda r:r["locale"] in {"de-DE","fr-FR"},
                     "decimal_point":lambda r:r["locale"] not in {"de-DE","fr-FR"}})
for name, keep in slice_checks.items():
    before=[r for r in sft if keep(r)]
    after=[r for r in rl if keep(r)]
    delta=100*(mean_field(after,"exact")-mean_field(before,"exact"))
    applies=len(before)>=30
    slice_deltas[name]={"n":len(before),"delta_pp":delta,"gate_applied":applies}
    if applies and delta < -GATES["slice_regression_pp"]:
        issues.append(f"slice regression: {name}")
if not rl_record["steps"]:
    issues.append("no nonzero rollout reward variation")
if not all(r["field_correct"]["currency"] for r in rl if r["guard"]):
    issues.append("wrong currency escaped the fixture check")
if not all(r["row_f1"] == 1 for r in rl if r["guard"]):
    issues.append("a missing or incorrect row escaped the fixture check")
decision="hold" if issues else "passes teaching quality gates"
write_results("15_release_results.json",{"decision":decision,"issues":issues,
    "release_split_sha256":test_hash,"base":baseline,"rules":rules,"sft":sft,"reward_adapter":rl,
    "paired_exact":paired_family_exact(sft,rl,lambda row:row["exact"]),
    "supervised_run":sft_record,"reward_run":rl_record,"external_headers":external,
    "external_revision":EXTERNAL_REV,"serving":serving,"slice_deltas":slice_deltas,
    "source_guard":{"controls":controls,"accepted":sum(r["guard"] for r in rl),"total":len(rl)},
    "candidate_sha256":candidate_sha,"incumbent_sha256":incumbent_sha,
    "rollback":{"matches":True,"key":fixture.key,"configuration":restored_config},
    "planned_thresholds":GATES,"t4_profile":profile})

{"elapsed_seconds": 3754.383, "gpu": "Tesla T4", "model": {"id": "Qwen/Qwen2-VL-2B-Instruct", "revision": "1860c46f12b7449d0f5b426a7474ab715342bc65"}, "packages": {"torch": "2.14.1+cu130", "transformers": "4.51.3", "peft": "0.15.2", "bitsandbytes": "0.50.2", "Pillow": "11.3.0", "datasets": "3.6.0"}, "catalogue": "course-invoice-v1", "seed": 42, "decision": "hold", "issues": ["exact_complete"], "release_split_sha256": "bd16e86b4b9ed1e63c4c3e93cca9e51bdde6746f7a04b19708471f23313614b1", "base": [{"key": "SF-750705", "family": "release-00", "layout": "packing", "locale": "en-US", "complete": true, "guard": false, "completion": "```json\n{\n  \"invoice_id\": 123456,\n  \"issued_on\": \"2022-04-15\",\n  \"supplier\": \"ABC Company\",\n  \"currency\": \"USD\",\n  \"line_items\": [\n    {\n      \"description\": \"Item 1\",\n      \"quantity\": 10,\n      \"unit_price\": 10.50,\n      \"unit_price\": 10.50,\n      \"subtotal\": 105.00,\n      \"tax\": 1.00,\n      \"total\": 106.00\n    },\n  

## Run it yourself

Report the release decision, the evidence behind it, and one failure that matters for real invoice users. Keep any decoding speedup separate from the model quality change.

Sources: [Qwen2-VL](https://huggingface.co/docs/transformers/v4.51.3/model_doc/qwen2_vl), [Generation strategies](https://huggingface.co/docs/transformers/v4.51.3/en/generation_strategies), [DeepSeekMath group-relative objective](https://arxiv.org/abs/2402.03300), [Invoice benchmark](https://huggingface.co/datasets/drew-ipp/invoice-extraction-benchmark).

[Read the result and release decision](https://docs.recohut.com/courses/advanced-llm-fine-tuning/4-release/15-release-a-document-parser).